In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
import sys

# sys.path.insert(0, "/home/matis/code/Florian-Q/maraicherbio-prediction/notebooks/")

In [5]:
import utils
import utils_series
import pandas as pd
import matplotlib.pyplot as plt

In [6]:
# --- Boucle : df_train / df_test pour chaque produit (split adaptatif, date fin commune) ---
df_all = utils.charger_dataframe()

# Définir la date de fin commune pour TOUS les splits
utils_series.GLOBAL_TEST_END_DATE = df_all['created'].max()
print(f'Date de fin commune : {utils_series.GLOBAL_TEST_END_DATE.date()}\n')

modeles = sorted(df_all['model'].unique())
print(f'{len(modeles)} produits à traiter\n')

dict_train = {}
dict_test = {}
skipped = []

for modele in modeles:
    try:
        df_produit = utils.charger_dataframe(modele)
        df_model = utils_series.complete_weekly_dataframe(df_produit, 'created', 'quantite_y')
        train, test = utils_series.split_adaptive_seasonal(df_model, test_pct=0.20)
        dict_train[modele] = train
        dict_test[modele] = test
    except ValueError as e:
        skipped.append(modele)
        continue

print(f'\nTerminé : {len(dict_train)} produits prêts  |  {len(skipped)} ignorés')
if skipped:
    print(f'Ignorés : {skipped}')

# Vérification
test_ends = [t.index.max().date() for t in dict_test.values()]
print(f'Toutes les fins de test = {test_ends[0]} ?  {len(set(test_ends)) == 1}')

Date de fin commune : 2026-05-27

100 produits à traiter

Train : 2014-06-08  →  2024-05-26  (521 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 12.0 ans (17%)
Train : 2014-07-06  →  2024-05-26  (517 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2014-06-29  →  2024-05-26  (518 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2015-01-11  →  2024-05-26  (490 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.4 ans (18%)
Train : 2018-05-06  →  2024-05-26  (317 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 8.1 ans (25%)
Train : 2014-01-05  →  2024-05-26  (543 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 12.4 ans (16%)
Train : 2014-02-23  →  2024-05-26  (536 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an

In [8]:
# ============================================================
# XGBOOST v1 : lag(y_t-1) + semaine ISO + split_adaptive + seasonal_metrics
# ============================================================
import numpy as np
import pandas as pd
import itertools
from xgboost import XGBRegressor

# ── Hyperparamètres fixes ─────────────────────────────────────────────────────
BEST_PARAMS: dict = {
    "subsample"        : 0.8,
    "colsample_bytree" : 0.8,
    "objective"        : "reg:squarederror",
    "random_state"     : 42,
    "verbosity"        : 0,
}

# ── Grid Search ───────────────────────────────────────────────────────────────
TUNING_GRID: dict = {
    "n_estimators"  : [100, 300],
    "max_depth"     : [3, 5],
    "learning_rate" : [0.05, 0.1],
}

ALL_COMBOS = list(itertools.product(
    TUNING_GRID["n_estimators"],
    TUNING_GRID["max_depth"],
    TUNING_GRID["learning_rate"],
))
# 2 × 2 × 2 = 8 combinaisons par produit

# ── Helpers ───────────────────────────────────────────────────────────────────

def _build_features(series: pd.Series) -> pd.DataFrame:
    """
    Construit le DataFrame de features à partir d'une Series hebdomadaire.
    Features : semaine ISO (1..53) + lag_1 (y_t-1).
    """
    df = pd.DataFrame({"y": series}, index=series.index)
    df["week"]  = df.index.isocalendar().week.astype(int)
    df["lag_1"] = df["y"].shift(1)
    return df


def _make_train_test_arrays(y_train: pd.Series, y_test: pd.Series):
    """
    Construit X_train, y_train_arr, X_test à partir des Series train/test.
    Le lag_1 en test utilise le dernier point connu du train.
    """
    # --- TRAIN ---
    df_train = _build_features(y_train)
    df_train = df_train.dropna()                      # retire la 1re ligne (lag_1 = NaN)
    X_train      = df_train[["week", "lag_1"]].values
    y_train_arr  = df_train["y"].values

    # --- TEST ---
    # On concatène train+test pour calculer lag_1 en continu,
    # puis on isole les lignes test
    full_series  = pd.concat([y_train, y_test])
    df_full      = _build_features(full_series)
    df_test      = df_full.loc[y_test.index]
    df_test      = df_test.fillna(0)                  # sécurité si jonction manquante
    X_test       = df_test[["week", "lag_1"]].values

    return X_train, y_train_arr, X_test


def _fit_predict_xgb(y_train: pd.Series, y_test: pd.Series,
                     n_est: int, depth: int, lr: float) -> np.ndarray:
    """
    Entraîne XGBRegressor sur y_train, prédit sur y_test.
    Retourne un tableau numpy clipé à 0.
    """
    X_train, y_train_arr, X_test = _make_train_test_arrays(y_train, y_test)

    model = XGBRegressor(
        n_estimators      = n_est,
        max_depth         = depth,
        learning_rate     = lr,
        subsample         = BEST_PARAMS["subsample"],
        colsample_bytree  = BEST_PARAMS["colsample_bytree"],
        objective         = BEST_PARAMS["objective"],
        random_state      = BEST_PARAMS["random_state"],
        verbosity         = BEST_PARAMS["verbosity"],
    )
    model.fit(X_train, y_train_arr)

    y_pred = np.clip(model.predict(X_test), 0, None)
    return y_pred


# ── Boucle principale ─────────────────────────────────────────────────────────
results = []

for modele in dict_train.keys():
    y_train = dict_train[modele]['quantite_y']
    y_test  = dict_test[modele]['quantite_y']

    best_mae  = np.inf
    best_pred = None
    best_nest = None
    best_dep  = None
    best_lr   = None

    # — Grid-search, optimisation sur MAE_all —
    for n_est, depth, lr in ALL_COMBOS:
        try:
            y_pred = _fit_predict_xgb(y_train, y_test, n_est, depth, lr)
            m_tmp  = utils_series.seasonal_metrics(y_test, y_pred)
            if m_tmp["MAPE"] < best_mae:
                best_mae  = m_tmp["MAPE"]
                best_pred = y_pred
                best_nest = n_est
                best_dep  = depth
                best_lr   = lr
        except Exception as e:
            print(f"[WARN] {modele} | n_est={n_est} depth={depth} lr={lr} → {e}")
            continue

    if best_pred is None:
        continue

    metrics = utils_series.seasonal_metrics(y_test, best_pred)

    results.append({
        'produit'    : modele,
        'train_debut': y_train.index.min().strftime('%Y-%m'),
        'test_fin'   : y_test.index.max().strftime('%Y-%m'),
        'train_sem'  : len(y_train),
        'test_sem'   : len(y_test),
        'best_nest'  : best_nest,
        'best_depth' : best_dep,
        'best_lr'    : best_lr,
        'pct_zeros'  : round(metrics['pct_zeros'], 1),
        'MAE_in'     : round(metrics['MAE_in'],    2),
        'MAE_out'    : round(metrics['MAE_out'],   2),
        'MAE_all'    : round(metrics['MAE_all'],   2),
        'MAPE'       : round(metrics['MAPE'],      1),
        'sMAPE_all'  : round(metrics['sMAPE_all'], 1),
    })

# ── Tableau ───────────────────────────────────────────────────────────────────
df_xgb = pd.DataFrame(results).sort_values('sMAPE_all')
print(f'XGBOOST v1 — {len(df_xgb)} produits '
      f'(date fin commune : {df_xgb["test_fin"].iloc[0]})\n')
print(df_xgb.to_string(index=False))

# ── Résumé ────────────────────────────────────────────────────────────────────
print(f'\nMAE_all moyen : {df_xgb["MAE_all"].mean():.2f}  |  '
      f'MAE_in moyen : {df_xgb["MAE_in"].mean():.2f}  |  '
      f'MAE_out moyen : {df_xgb["MAE_out"].mean():.2f}')
print(f'sMAPE médian  : {df_xgb["sMAPE_all"].median():.1f} %  |  '
      f'Zéros moyen  : {df_xgb["pct_zeros"].mean():.1f} %')

# ── Top 5 / Bottom 5 ──────────────────────────────────────────────────────────
print(f'\n--- Top 5 (sMAPE) ---')
print(df_xgb[['produit', 'MAE_all', 'MAPE', 'sMAPE_all',
              'best_nest', 'best_depth', 'best_lr']].head(5).to_string(index=False))
print(f'\n--- Bottom 5 (sMAPE) ---')
print(df_xgb[['produit', 'MAE_all', 'MAPE', 'sMAPE_all',
              'best_nest', 'best_depth', 'best_lr']].tail(5).to_string(index=False))

XGBOOST v1 — 84 produits (date fin commune : 2026-05)

                      produit train_debut test_fin  train_sem  test_sem  best_nest  best_depth  best_lr  pct_zeros  MAE_in  MAE_out  MAE_all  MAPE  sMAPE_all
                        Prune     2019-07  2026-05        306        52        100           3     0.05       82.7    3.68     0.00     0.64  53.1       11.8
              tomate ancienne     2019-08  2026-05        304        52        100           5     0.05       63.5    2.84     0.06     1.08  43.3       22.7
                 Tomate paola     2014-07  2026-05        516       104        100           5     0.10       63.5    7.66     0.03     2.82  87.9       22.8
                 patate douce     2020-10  2026-05        243        52        100           3     0.05       73.1    3.24     0.20     1.02  49.7       23.4
                         Fève     2015-06  2026-05        467       105        100           3     0.05       83.8    3.43     0.19     0.71  48.4       25

In [1]:
df_xgb['MAPE'].mean()

NameError: name 'df_xgb' is not defined

In [14]:
df_xgb.to_csv('../data/XGboost_metrics.csv', index=False)